## Preparando o ambiente
Instalamos as bibliotecas com `pip`. O `requests` faz as requisições HTTP; o `beautifulsoup4` (com o motor `lxml`) organiza o HTML.


In [1]:
!pip install -q requests beautifulsoup4 lxml
print('Instalação feita com sucesso')


Instalação feita com sucesso


## Parte 1 — Baixando o HTML com `requests`


In [3]:
import requests

URL = "https://books.toscrape.com/"

# Faz a requisição GET (pede o conteúdo da página)
res = requests.get(URL)

print("status_code:", res.status_code)          # 200 = ok
print("Caracteres no res.text:", len(res.text))  # tamanho do HTML

# Mostra apenas os primeiros 30000 caracteres do HTML cru
print("\n----- HTML CRU (início) -----")
print(res.text[:30000])

status_code: 200
Caracteres no res.text: 51294

----- HTML CRU (início) -----
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-

## Parte 2 — Extraindo os dados com `BeautifulSoup`


In [5]:
from bs4 import BeautifulSoup

# Entrega o HTML (baixado com requests) ao BeautifulSoup
soup = BeautifulSoup(res.text, "lxml")

# find: primeiro elemento. Aqui, o título principal da página (<h1>)
titulo = soup.find("title")
print("Título da página:", titulo.get_text(strip=True))

Título da página: All products | Books to Scrape - Sandbox


In [22]:
# find_all: Encontra todas as caixas de livros na primeira página
caixas = soup.find_all("article", class_="product_pod")
print("Livros encontrados na primeira página:", len(caixas))

Livros encontrados na primeira página: 20


In [41]:
# Para cada caixa, extraímos:   nome do livro + preco + estoque + avaliacao
dados = []
for caixa in caixas:
    nome      = caixa.find("h3").find("a").get_text(strip=True) #mas se quiser que apareça o nome COMPLETO: caixa.find("h3").find("a")["title"]
    preco     = caixa.find("p", class_="price_color").get_text(strip=True)
    estoque   = caixa.find("p", class_="instock availability").get_text(strip=True)
    avaliacao = caixa.find("p", class_="star-rating")["class"][1]
    
    dados.append({"nome": nome,"preco": preco,"avaliacao": avaliacao,})

    print(f"{nome:20} | {preco[:20]} | {estoque[:20]} | {avaliacao[:20]}") 

A Light in the ...   | Â£51.77 | In stock | Three
Tipping the Velvet   | Â£53.74 | In stock | One
Soumission           | Â£50.10 | In stock | One
Sharp Objects        | Â£47.82 | In stock | Four
Sapiens: A Brief History ... | Â£54.23 | In stock | Five
The Requiem Red      | Â£22.65 | In stock | One
The Dirty Little Secrets ... | Â£33.34 | In stock | Four
The Coming Woman: A ... | Â£17.93 | In stock | Three
The Boys in the ...  | Â£22.60 | In stock | Four
The Black Maria      | Â£52.15 | In stock | One
Starving Hearts (Triangular Trade ... | Â£13.99 | In stock | Two
Shakespeare's Sonnets | Â£20.66 | In stock | Four
Set Me Free          | Â£17.46 | In stock | Five
Scott Pilgrim's Precious Little ... | Â£52.29 | In stock | Five
Rip it Up and ...    | Â£35.02 | In stock | Five
Our Band Could Be ... | Â£57.25 | In stock | Three
Olio                 | Â£23.88 | In stock | One
Mesaerion: The Best Science ... | Â£37.59 | In stock | One
Libertarianism for Beginners | Â£51.33 | In stock | Two
It

## Percorrendo várias páginas (paginação)

Um **crawler** simples: o catálogo tem várias páginas no padrão
`catalogue/page-N.html`. Vamos percorrer as 3 primeiras e juntar tudo.

> Boas práticas: usamos `time.sleep` entre as requisições para não sobrecarregar
> o servidor.


In [45]:
import time

todos = []
for pagina in range(1, 4):  # páginas 1, 2 e 3
    url = f"https://books.toscrape.com/catalogue/page-{pagina}.html"
    r = requests.get(url)
    if r.status_code != 200:
        break
        
    s = BeautifulSoup(r.text, "lxml")
    for caixa in s.find_all("article", class_="product_pod"):
        todos.append({
            "titulo": caixa.find("h3").find("a")["title"],
            "preco": caixa.find("p", class_="price_color").get_text(strip=True),
            "nota": caixa.find("p", class_="star-rating")["class"][1],
        })
        
    print(f"Página {pagina}: total acumulado = {len(todos)} livros")
    time.sleep(1)  # pausa educada entre as requisições

Página 1: total acumulado = 20 livros
Página 2: total acumulado = 40 livros
Página 3: total acumulado = 60 livros


### Tabela


In [47]:
import pandas as pd

df = pd.DataFrame(dados)
print(df)

# Salvando em CSV (aparece no painel de arquivos do Colab)
df.to_csv("books.csv", index=False)
print("Arquivo books.csv salvo!")

                                     nome    preco avaliacao
0                      A Light in the ...  Â£51.77     Three
1                      Tipping the Velvet  Â£53.74       One
2                              Soumission  Â£50.10       One
3                           Sharp Objects  Â£47.82      Four
4            Sapiens: A Brief History ...  Â£54.23      Five
5                         The Requiem Red  Â£22.65       One
6            The Dirty Little Secrets ...  Â£33.34      Four
7                 The Coming Woman: A ...  Â£17.93     Three
8                     The Boys in the ...  Â£22.60      Four
9                         The Black Maria  Â£52.15       One
10  Starving Hearts (Triangular Trade ...  Â£13.99       Two
11                  Shakespeare's Sonnets  Â£20.66      Four
12                            Set Me Free  Â£17.46      Five
13    Scott Pilgrim's Precious Little ...  Â£52.29      Five
14                      Rip it Up and ...  Â£35.02      Five
15                  Our 